# Module 8: Feature Views

**Snowpark ML Fundamentals - Week 2 | Lunch & Learn**

---

## Learning Objectives
- Initialize the Snowflake Feature Store
- Create and register Entities (join keys)
- Create managed vs external FeatureViews
- Register FeatureViews for training and inference

## Key Concept
> **Managed FeatureViews** are computed and refreshed by Snowflake on a schedule.
> **External FeatureViews** wrap pre-computed tables (e.g., from dbt) with no refresh schedule.

---

In [ ]:
%load_ext autoreload
%autoreload 2

## 8.1 Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_rfm_features,
    create_customer_interactions_dataset,
    create_behavioral_features,
)

session = create_session(env_path='../.env')

# Ensure feature tables exist
create_customer_transactions_dataset(session)
create_customer_interactions_dataset(session)
rfm_df = create_rfm_features(session)
behavioral_df = create_behavioral_features(session)
print("Feature tables ready.")

## 8.2 Initialize the Feature Store

In [ ]:
from snowpark_fundamentals.feature_store.entities import (
    setup_feature_store,
    create_customer_entity,
    register_entity,
)

# Initialize the Feature Store in the current database/schema
fs = setup_feature_store(session)
print(f"Feature Store initialized")

## 8.3 Create and Register an Entity

An **Entity** defines the join key(s) for a group of features.
Think of it as the grain — `CUSTOMER_ID` means "one row per customer."

In [ ]:
# Create and register the Customer entity
customer_entity = create_customer_entity()
registered_entity = register_entity(fs, customer_entity)
print(f"Entity registered: {customer_entity.name}")
print(f"Join keys: {customer_entity.join_keys}")

## 8.4 Create FeatureViews

There are two types of FeatureViews:

| Type | `refresh_freq` | Backend | Use case |
|------|---------------|---------|----------|
| **Managed** | `"1 day"`, `"1 hour"` | Dynamic Table (auto-refreshed) | Raw data that Snowflake computes |
| **External** | `None` | View (no refresh) | Pre-computed tables from dbt/Airflow |

**Managed FeatureViews** require the warehouse to have `OPERATE` privilege for Dynamic Table
creation. For this tutorial we use **external FeatureViews** since our feature tables are
already pre-computed (notebook 07).

In production, if you want Snowflake to own the refresh:
```python
rfm_fv = create_managed_feature_view(
    name="CUSTOMER_RFM_FV",
    entities=[customer_entity],
    feature_df=rfm_query,
    refresh_freq="1 day",       # <-- Snowflake creates a Dynamic Table
    timestamp_col="_FEATURE_TIMESTAMP",
)
```

In [ ]:
from snowpark_fundamentals.feature_store.feature_views import (
    create_external_feature_view,
    register_feature_view,
)

# RFM features — pre-computed in notebook 07, registered as external FeatureView
db = session.get_current_database().replace('"', '')
schema = session.get_current_schema().replace('"', '')
rfm_query = session.table(f"{db}.{schema}.CUSTOMER_RFM_FEATURES")

rfm_fv = create_external_feature_view(
    name="CUSTOMER_RFM_FV",
    entities=[customer_entity],
    feature_df=rfm_query,
    desc="RFM features: recency, frequency, monetary with 30d/90d/365d windows",
    timestamp_col="_FEATURE_TIMESTAMP",
)

registered_rfm = register_feature_view(fs, rfm_fv, version="V1", overwrite=True)
print("External FeatureView registered: CUSTOMER_RFM_FV V1")

In [ ]:
# Inspect the registered RFM FeatureView
print(f"Name:        {registered_rfm.name}")
print(f"Version:     {registered_rfm.version}")
print(f"Status:      {registered_rfm.status}")
print(f"Description: {registered_rfm.desc}")
print(f"\nFeature columns:")
registered_rfm.feature_df.show(5)

## 8.5 Create a Second External FeatureView

Let's register a second feature set — behavioral engagement features.
Both FeatureViews share the same `CUSTOMER` entity (same grain).

In [ ]:
# Behavioral features — imagine these are computed by dbt
behavioral_query = session.table(f"{db}.{schema}.CUSTOMER_BEHAVIORAL_FEATURES")

behavioral_fv = create_external_feature_view(
    name="CUSTOMER_BEHAVIORAL_FV",
    entities=[customer_entity],
    feature_df=behavioral_query,
    desc="Behavioral engagement features (managed by dbt)",
    timestamp_col="_FEATURE_TIMESTAMP",
)

registered_behavioral = register_feature_view(fs, behavioral_fv, version="V1", overwrite=True)
print("External FeatureView registered: CUSTOMER_BEHAVIORAL_FV V1")

In [ ]:
# Inspect the registered Behavioral FeatureView
print(f"Name:        {registered_behavioral.name}")
print(f"Version:     {registered_behavioral.version}")
print(f"Status:      {registered_behavioral.status}")
print(f"Description: {registered_behavioral.desc}")
print(f"\nFeature columns:")
registered_behavioral.feature_df.show(5)

In [ ]:
from snowpark_fundamentals.feature_store.feature_views import list_feature_views
from snowpark_fundamentals.feature_store.entities import list_entities

# All registered entities
print("=== Registered Entities ===")
list_entities(fs).show()

# All registered feature views
print("\n=== Registered FeatureViews ===")
list_feature_views(fs).show()

## 8.6 Managed vs External: Comparison

| Aspect | Managed | External |
|--------|---------|----------|
| Refresh | Snowflake schedules (e.g., daily) | You manage (dbt, Airflow) |
| `refresh_freq` | Required (e.g., `'1 day'`) | `None` |
| Best for | Simple aggregations from raw tables | dbt output, complex pipelines |
| Compute cost | Snowflake warehouse on schedule | Your pipeline's compute |

## Key Takeaways

1. **Entities** define the grain (join keys) for feature lookups
2. **Managed FeatureViews** let Snowflake handle refresh scheduling
3. **External FeatureViews** integrate with dbt or other orchestration tools
4. Registration makes features available for training sets and inference

---

**Next: [09 - Training Sets](09_training_sets.ipynb)**

In [ ]:
session.close()